# 07 — Threads, Locks, and Queues

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
# Provided for this lesson.
from collections.abc import Callable

from lesson import Exercises, ___, run

check = Exercises('advanced/07', {1: '701707a4747c962d', 2: '951be31d9f597680', 3: 'aabae8a772e54f37', 4: '701acfaf6e3a2f29', 5: '3cab66c4a9562c6b', 6: '72c9943890427cac', 7: '3a0246f6f9032668'}, practice='advanced_02_concurrency_resilience', independent='map_threaded')

## 1. Thread pools

- `ThreadPoolExecutor(max_workers=n)` runs functions on a pool of threads; good for waiting on I/O.
- `pool.map(func, items)` returns results in input order.
- Threads aren't available in the browser runtime; run these on your computer or Colab.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=4) as pool:
    print(list(pool.map(str.upper, ["a", "b", "c"])))

**Exercise 1.** `pool.map` returns results in which order? Write `"input"` or `"finish"`.

In [ ]:
answer = ___
check(1, answer)

**Exercise 2.** Which kind of work do threads speed up in CPython: `"io"` or `"cpu"`?

In [ ]:
answer = ___
check(2, answer)

## 2. Locks

- `count += 1` is read-modify-write: two threads can interleave and lose updates.
- Guard shared state with `threading.Lock()`: `with self._lock: self._value += 1`.
- Keep the locked part short and never call unknown code while holding a lock.

In [ ]:
import threading
class SafeCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()
    def increment(self):
        with self._lock:
            self._value += 1
    @property
    def value(self):
        return self._value

c = SafeCounter()
for _ in range(3):
    c.increment()
print(c.value)

**Exercise 3.** Which `threading` class makes a section mutually exclusive? Write its name.

In [ ]:
answer = ___
check(3, answer)

**Exercise 4.** Use the lock as a context manager while incrementing.

In [ ]:
import threading
lock = threading.Lock()
count = 0
for _ in range(5):
    with ___:
        count += 1
check(4, count)

**Exercise 5.** Is a lock released if the code inside `with lock:` raises? Write `True` or `False`.

In [ ]:
answer = ___
check(5, answer)

## 3. Thread-safe queues

- `queue.Queue` is safe to share between threads: producers `put`, consumers `get`.
- It's first in, first out. `queue.LifoQueue` and `queue.PriorityQueue` change the order.
- Signal the end with a sentinel such as `None`.

In [ ]:
import queue
q = queue.Queue()
for item in ["a", "b", None]:
    q.put(item)
out = []
while (item := q.get()) is not None:
    out.append(item)
print(out)

**Exercise 6.** Put 1, 2, 3 into a `queue.Queue` and get one. Which comes out?

In [ ]:
predicted = ___
check(6, predicted)

**Exercise 7.** Drain the supplied work queue in retrieval order using get_nowait. A caller waits for task completion, so acknowledge every retrieved item even if visit raises. Propagate the failure and leave later items queued.

In [ ]:
def consume(queue, visit):
    raise NotImplementedError

def observe(drain):
    from queue import Queue
    q = Queue()
    for value in (1, 2, 3):
        q.put(value)
    seen = []
    def fail_on_two(value):
        seen.append(value)
        if value == 2:
            raise ValueError("bad item")
    try:
        drain(q, fail_on_two)
    except ValueError:
        outstanding = q.unfinished_tasks
        drain(q, seen.append)
        return seen, outstanding, q.unfinished_tasks
    return "failure swallowed"

check(7, consume, observe=observe)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `SafeCounter`

A counter that is safe to increment from many threads.

In [ ]:
class SafeCounter:
    """A counter that is safe to increment from many threads (threading.Lock)."""

    def __init__(self) -> None:
        self._value = 0
        self._lock = ___

    def increment(self) -> None:
        with ___:
            self._value += 1

    @property
    def value(self) -> int:
        return ___

In [ ]:
check('SafeCounter')

### Independent build — `map_threaded`

Apply func to each item in a thread pool of at most workers threads. Return results in input order. Empty input gives []. Propagate worker errors to the caller.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def map_threaded(func: Callable, items: list, workers: int = 4) -> list:
    raise NotImplementedError

In [ ]:
check('map_threaded')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Is `counter += 1` safe from several threads without a lock?

   - **a)** No, it's a read-modify-write that can interleave
   - **b)** Only on Linux
   - **c)** Yes, the GIL protects it

**2.** queue.Queue is useful because it…

   - **a)** Is thread-safe for producers and consumers
   - **b)** Runs code faster
   - **c)** Avoids threads

**3.** Threads help most with…

   - **a)** I/O-bound waiting
   - **b)** Neither
   - **c)** CPU-bound pure Python

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/07', {1: 'e7117ec4e8e6', 2: '35d5e60d3456', 3: '5a3855e387e1'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [08 — Multiprocessing, Process Pools, and the GIL](08_multiprocessing_and_parallelism.ipynb)